# bootstrap_student

> 对应代码：`EconometricsCode/code_in_notes/Chap.9/bootstrap_student.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。


首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.9`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [ ]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.9")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


清空内存后，我们读入俄勒冈医保实验（OHIE）数据，后面将在它上面演示学生化 bootstrap（percentile-t）置信区间的构造。

In [ ]:
%%stata
clear all
use ../datasets/OHIE_QJE.dta


这是本文件的核心命令。bootstrap 前缀对冒号后的回归重复执行 200 次（reps(200)）：每次按 household_id 整簇重抽样（cluster(household_id)，即聚类 bootstrap，保持同一家庭内部观测的相关结构不被抽样破坏），重新估计回归并记录 treatment 的系数 b 与标准误 se；saving() 把 200 组 (b, se) 存入 bootstrap_student.dta，seed(55) 固定随机数保证可复现。学生化 bootstrap 的关键在于：我们不仅保存每次抽样的系数，还保存每次抽样内的标准误，从而可以构造每个重复样本的 t 统计量。

In [ ]:
%%stata
bootstrap b=_b[treatment] se=_se[treatment], reps(200) seed(55) cluster(household_id) saving(bootstrap_student.dta, replace): reg birthyear_list treatment, vce(bootstrap)


先在原始样本上跑一次回归，把 treatment 的点估计 b̂ 与标准误 se 存入局部宏，作为后面学生化统计量的中心与尺度。

In [ ]:
%%stata
reg birthyear_list treatment
local b=_b[treatment]
local se=_se[treatment]


用 frame 新开一个数据框装入刚才保存的 bootstrap 抽样结果，避免覆盖内存中的原始数据。

In [ ]:
%%stata
frame create bootstrap_sample
frame change bootstrap_sample
use bootstrap_student.dta


对每个 bootstrap 重复样本计算学生化统计量 t=(b−b̂)/se：以原始样本的点估计为中心、用各次抽样自己的标准误标准化。排序后取 2.5% 与 97.5% 分位数（用 floor/ceil 取相邻两个次序统计量的平均做插值）。这两个分位数 α_0.025、α_0.975 就是用来代替正态临界值 ±1.96 的 bootstrap 临界值。

In [ ]:
%%stata
gen t=(b-`b')/se
sort t
local alpha_25=(t[floor(_N*0.025)]+t[ceil(_N*0.025)])/2
local alpha_975=(t[floor(_N*0.975)]+t[ceil(_N*0.975)])/2


分位数已存入局部宏，切回默认数据框并删除临时文件 bootstrap_student.dta。

In [ ]:
%%stata
frame change default
rm bootstrap_student.dta


输出 percentile-t 置信区间 [b̂−α_0.975·se, b̂−α_0.025·se]。注意端点的不对称性：左端点减的是 α_0.975（t 分布的右尾），右端点减的是 α_0.025（t 分布的左尾）。这正是学生化 bootstrap 的思想——把 t=(b̂−β)/se 当作近似枢轴量，用 bootstrap 分布的分位数而非正态临界值来构造区间，对偏态与小样本具有更准确的覆盖率。

In [ ]:
%%stata
di "置信区间为：" _skip "[" `b'-`alpha_975'*`se' "," `b'-`alpha_25'*`se' "]"
